# R3 Voucher Chain — IV Surface, Smile Drift, Static-vs-Rolling

Rich EDA notebook for the VELVETFRUIT_EXTRACT (VFE) voucher chain in Prosperity 4 R3.
Source script: `notebooks/03_voucher_chain_eda.py`. Findings doc: `docs/round_3/research/05_voucher_chain.md`.

**Conventions**
- Moneyness: `m = log(K/S) / sqrt(TTE_y)`, year basis = 365.
- TTE: 8 / 7 / 6 days for historical day 0 / 1 / 2.
- Smile: `iv = a*m^2 + b*m + c` (CMU convention).
- Position limit per voucher = 300; VFE limit = 200.
- Skip vouchers pinned at mid==0.5 (deep-OTM platform floor) when fitting.

Run all cells. Final cell = TL;DR + every numeric finding from the .md doc.

## Setup

Imports, paths, constants, and Black-Scholes helper. `%matplotlib inline` for rendered plots in Jupyter.

In [ ]:
%matplotlib inline
from __future__ import annotations
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
from utils.black_scholes import BlackScholes  # noqa: E402

PLOTS = os.path.join(ROOT, "docs/round_3/research/plots")
os.makedirs(PLOTS, exist_ok=True)

VOUCHERS = ["VEV_4000", "VEV_4500", "VEV_5000", "VEV_5100", "VEV_5200",
            "VEV_5300", "VEV_5400", "VEV_5500", "VEV_6000", "VEV_6500"]
STRIKES = {v: int(v.split("_")[1]) for v in VOUCHERS}
DEAD = {"VEV_6000", "VEV_6500"}
UND = "VELVETFRUIT_EXTRACT"
TTE_DAYS_BY_DAY = {0: 8, 1: 7, 2: 6}  # historical TTE per brief

## Load voucher panel

Concat the three day CSVs (semicolon-separated), pivot to wide form (one row per `(day, timestamp)` with mid for each product), and attach `S` and TTE columns.

In [ ]:
def load() -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(ROOT, "data/round_3/prices_round_3_day_*.csv")))
    df = pd.concat([pd.read_csv(f, sep=";") for f in files], ignore_index=True)
    df["t"] = df.day * 1_000_000 + df.timestamp
    return df

print("Loading data ...")
raw = load()
print("rows:", len(raw), "products:", raw["product"].nunique())

# Wide table: per-(day,timestamp) one row with mid for each product
wide = raw.pivot_table(index=["day", "timestamp"], columns="product",
                       values="mid_price").reset_index()
wide = wide.dropna(subset=[UND]).copy()
wide["S"] = wide[UND]
wide["tte_days"] = wide["day"].map(TTE_DAYS_BY_DAY)
wide["tte_y"] = wide["tte_days"] / 365.0

print("\nVFE summary by day:")
print(wide.groupby("day")["S"].agg(["mean", "std", "min", "max"]).round(2))

## Solve implied vol per (timestamp, voucher)

Bisection IV solve via `BlackScholes.implied_vol`. Drop rows where mid is below intrinsic, hit bisection bounds, or the voucher is the dead-pinned 0.5 floor. Also compute `m` (moneyness), vega, and delta for downstream analysis.

In [ ]:
print("Solving IV per (timestamp, voucher) ...")
records = []
for _, row in wide.iterrows():
    S = row["S"]; T = row["tte_y"]
    for v in VOUCHERS:
        mid = row.get(v, np.nan)
        if pd.isna(mid) or mid <= 0:
            continue
        K = STRIKES[v]
        intrinsic = max(S - K, 0.0)
        # Filter: cannot solve IV if mid below intrinsic or essentially 0
        if mid < intrinsic + 1e-6 or mid <= 0.5 + 1e-9 and v in DEAD:
            iv = np.nan
        else:
            try:
                iv = BlackScholes.implied_vol(mid, S, K, T)
                # bisection bounds 0.001..1.0; treat hit-bound as bad
                if iv <= 0.0015 or iv >= 0.999:
                    iv = np.nan
            except Exception:
                iv = np.nan
        m = np.log(K / S) / np.sqrt(T)
        if not np.isnan(iv):
            vega = BlackScholes.vega(S, K, T, iv)
            delta = BlackScholes.delta(S, K, T, iv)
        else:
            vega = np.nan; delta = np.nan
        records.append((row["day"], row["timestamp"], v, K, S, T, mid, m, iv, vega, delta))

iv_df = pd.DataFrame(records, columns=["day", "timestamp", "voucher", "K", "S",
                                       "T", "mid", "m", "iv", "vega", "delta"])
print("IV rows:", len(iv_df), "non-null IV:", iv_df["iv"].notna().sum())

## Q1 — Per-voucher IV summary + surface

Distribution stats per voucher, IV time series per strike (3 days overlaid), and a (day x voucher) mean-IV heatmap. Identifies which strikes give clean IV solutions.

In [ ]:
print("=== Q1: per-voucher IV summary ===")
summary = iv_df.groupby("voucher")["iv"].agg(["count", "mean", "std", "min", "max"]).round(4)
print(summary)

# Plot: IV distribution and time series per voucher
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=False)
for ax, v in zip(axes.flat, VOUCHERS):
    sub = iv_df[iv_df.voucher == v].dropna(subset=["iv"])
    if len(sub) == 0:
        ax.set_title(f"{v} (no IV)"); continue
    for d, g in sub.groupby("day"):
        ax.plot(g["timestamp"], g["iv"], lw=0.4, label=f"d{d}")
    ax.set_title(v); ax.set_xlabel("ts"); ax.set_ylabel("iv"); ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig(f"{PLOTS}/03_iv_timeseries.png", dpi=110); plt.show()

# IV surface: mean IV per (day, voucher) heatmap
piv = iv_df.dropna(subset=["iv"]).groupby(["day", "voucher"])["iv"].mean().unstack()
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(piv.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=45)
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels([f"day{d}" for d in piv.index])
ax.set_title("Mean implied vol — IV surface")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(f"{PLOTS}/03_iv_surface.png", dpi=110); plt.show()
print("\nMean IV per (day, voucher):"); print(piv.round(4))

## Q2 — Per-tick quadratic smile fit + intraday drift

Fit `iv = a*m^2 + b*m + c` at each `(day, timestamp)` over live strikes (excluding dead 6000/6500). Plot `a(t), b(t), c(t)` per day and quantify drift via `|std/mean|`. Curvature `a` is the CMU smoking gun.

In [ ]:
print("=== Q2: per-timestamp quadratic smile fit ===")
fit_df = iv_df.dropna(subset=["iv"]).copy()
# exclude DEAD strikes from fitting (no real market)
fit_df = fit_df[~fit_df.voucher.isin(DEAD)]

def fit_quadratic(g):
    if len(g) < 4:
        return pd.Series({"a": np.nan, "b": np.nan, "c": np.nan, "n": len(g)})
    x = g["m"].values; y = g["iv"].values
    coef = np.polyfit(x, y, 2)  # returns [a, b, c]
    return pd.Series({"a": coef[0], "b": coef[1], "c": coef[2], "n": len(g)})

smile = fit_df.groupby(["day", "timestamp"]).apply(fit_quadratic).reset_index()
print("Smile fits:", len(smile), "valid:", smile["a"].notna().sum())
print("\nSmile coefficient stats by day:")
print(smile.groupby("day")[["a", "b", "c"]].agg(["mean", "std", "min", "max"]).round(4))

# Plot a(t), b(t), c(t) per day
fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharex="col")
for col, d in enumerate(sorted(smile["day"].unique())):
    sub = smile[smile.day == d]
    for row, k in enumerate(["a", "b", "c"]):
        ax = axes[row, col]
        ax.plot(sub["timestamp"], sub[k], lw=0.5)
        ax.axhline(sub[k].mean(), color="red", ls="--", lw=0.7,
                   label=f"mean={sub[k].mean():.3f}")
        ax.set_title(f"day{d} — {k}(t)")
        ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{PLOTS}/03_smile_drift.png", dpi=110); plt.show()

# Quantify drift: stdev / mean ratio per day
drift = (smile.groupby("day")[["a", "b", "c"]].std()
         / smile.groupby("day")[["a", "b", "c"]].mean().abs()).round(3)
print("\nIntraday |std/mean| per day (drift signature):")
print(drift)

## Q3 — Static day-0 smile vs out-of-sample (days 1, 2)

Fit one pooled smile over all of day 0, then price every voucher on days 1 and 2 with it. Per-(day, voucher) residual mean tells us whether a Frankfurt-style hardcoded smile generalises.

In [ ]:
print("=== Q3: static (day-0 fit) vs out-of-sample days 1,2 ===")
day0 = fit_df[fit_df.day == 0]
coef0 = np.polyfit(day0["m"].values, day0["iv"].values, 2)
print(f"Day-0 pooled smile: a={coef0[0]:.4f}, b={coef0[1]:.4f}, c={coef0[2]:.4f}")

def predict_iv(m, coef):
    return coef[0]*m*m + coef[1]*m + coef[2]

def bs_price_safe(S, K, T, iv):
    if not np.isfinite(iv) or iv <= 0:
        return np.nan
    return BlackScholes.call_price(S, K, T, iv)

# residual = market_mid - theo on each day using static day-0 smile
fit_df["iv_static"] = predict_iv(fit_df["m"].values, coef0)
# NOTE: fit_df.T is the DataFrame transpose; use fit_df["T"] for the TTE column.
fit_df["theo_static"] = [bs_price_safe(s, k, t, iv) for s, k, t, iv
                         in zip(fit_df["S"], fit_df["K"], fit_df["T"], fit_df["iv_static"])]
fit_df["resid_static"] = fit_df["mid"] - fit_df["theo_static"]
print("\nStatic-smile residual (mid - theo) by day, voucher:")
print(fit_df.groupby(["day", "voucher"])["resid_static"]
        .agg(["mean", "std"]).round(3))

## Q4 — Rolling smile (window=100 ticks) vs static

Rolling-mean the per-tick `(a, b, c)` over 100 ticks (`min_periods=20`), reprice each voucher with the rolling smile, and compare RMSE per voucher.

In [ ]:
print("=== Q4: rolling smile (window=100 ticks) ===")
WINDOW = 100
# Build per-tick rolling smile by reusing per-tick fits → take rolling mean coefs
smile_sorted = smile.sort_values(["day", "timestamp"]).reset_index(drop=True)
smile_sorted[["a_roll", "b_roll", "c_roll"]] = (
    smile_sorted[["a", "b", "c"]].rolling(WINDOW, min_periods=20).mean())

# Residuals using rolling smile (predicting next tick's voucher mid)
fit_df = fit_df.merge(smile_sorted[["day", "timestamp", "a_roll", "b_roll", "c_roll"]],
                      on=["day", "timestamp"], how="left")
fit_df["iv_roll"] = (fit_df["a_roll"]*fit_df["m"]**2
                     + fit_df["b_roll"]*fit_df["m"]
                     + fit_df["c_roll"])
fit_df["theo_roll"] = [bs_price_safe(s, k, t, iv) for s, k, t, iv
                       in zip(fit_df["S"], fit_df["K"], fit_df["T"], fit_df["iv_roll"])]
fit_df["resid_roll"] = fit_df["mid"] - fit_df["theo_roll"]

cmp = pd.DataFrame({
    "static_rmse": fit_df.groupby("voucher")["resid_static"].apply(lambda s: np.sqrt((s**2).mean())),
    "rolling_rmse": fit_df.groupby("voucher")["resid_roll"].apply(lambda s: np.sqrt((s**2).mean())),
})
cmp["improvement_%"] = (1 - cmp.rolling_rmse / cmp.static_rmse) * 100
print("\nRMSE per voucher (lower=better):")
print(cmp.round(3))

overall_static = np.sqrt((fit_df["resid_static"]**2).mean())
overall_roll = np.sqrt((fit_df["resid_roll"]**2).mean())
print(f"\nOverall RMSE: static={overall_static:.3f}, rolling={overall_roll:.3f}, "
      f"improvement={(1-overall_roll/overall_static)*100:.1f}%")

## Q5 — Residual stationarity (ADF) + per-strike residual plots

Augmented Dickey-Fuller on the rolling-smile residual per voucher. Stationary residuals = `mid - theo` mean-reverts = tradable contrarian alpha.

In [ ]:
print("=== Q5: residual stationarity / mean-reversion ===")
from statsmodels.tsa.stattools import adfuller
adf_rows = []
for v in [v for v in VOUCHERS if v not in DEAD]:
    s = fit_df[fit_df.voucher == v]["resid_roll"].dropna()
    if len(s) < 50:
        continue
    try:
        stat, pval, *_ = adfuller(s.values, maxlag=10)
        adf_rows.append((v, len(s), stat, pval, s.mean(), s.std()))
    except Exception as e:
        adf_rows.append((v, len(s), np.nan, np.nan, s.mean(), s.std()))
adf_df = pd.DataFrame(adf_rows, columns=["voucher", "n", "adf_stat", "pval", "mean", "std"])
print(adf_df.round(4))

# Plot residuals per strike (rolling smile)
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=False)
live = [v for v in VOUCHERS if v not in DEAD]
for ax, v in zip(axes.flat, live):
    sub = fit_df[fit_df.voucher == v].dropna(subset=["resid_roll"])
    for d, g in sub.groupby("day"):
        ax.plot(g["timestamp"], g["resid_roll"], lw=0.4, label=f"d{d}")
    ax.axhline(0, color="k", lw=0.4)
    ax.set_title(f"{v} resid (mid-theo, rolling smile)")
    ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig(f"{PLOTS}/03_residuals_per_strike.png", dpi=110); plt.show()

## Q6 — Pairwise cointegration on voucher mids (Engle-Granger)

Run the two-step cointegration test on every live-voucher pair. Strong cointegration supports cross-strike relative-value trades independent of underlying direction.

In [ ]:
print("=== Q6: pairwise cointegration (Engle-Granger) on voucher mids ===")
from statsmodels.tsa.stattools import coint
live = [v for v in VOUCHERS if v not in DEAD]
mid_wide = (raw[raw["product"].isin(live)]
            .pivot_table(index=["day", "timestamp"], columns="product",
                         values="mid_price").dropna())
coint_rows = []
pairs = [(a, b) for i, a in enumerate(live) for b in live[i+1:]]
for a, b in pairs:
    if a not in mid_wide or b not in mid_wide:
        continue
    x = mid_wide[a].values; y = mid_wide[b].values
    if np.std(x) == 0 or np.std(y) == 0:
        continue
    try:
        t, p, _ = coint(x, y)
        coint_rows.append((a, b, t, p))
    except Exception:
        pass
coint_df = pd.DataFrame(coint_rows, columns=["a", "b", "t", "p"]).sort_values("p")
print(coint_df.head(15).round(4))
print(f"\nPairs with p<0.05: {(coint_df.p < 0.05).sum()} / {len(coint_df)}")

## Q7 — Vega-weighted residuals (Frankfurt-style threshold prep)

Divide residuals by vega to get a strike-comparable signal scale. Use to set a uniform `|resid| > k * vega` entry threshold across the chain.

In [ ]:
print("=== Q7: vega-weighted residuals (Frankfurt threshold candidates) ===")
fit_df["resid_per_vega"] = fit_df["resid_roll"] / fit_df["vega"]
vw = fit_df.groupby("voucher")[["vega", "resid_roll", "resid_per_vega"]].agg(
    {"vega": "mean", "resid_roll": ["mean", "std"], "resid_per_vega": ["mean", "std"]})
print(vw.round(4))

## Q8 — Implied vs realised vol on VFE

Compute annualised realised vol of VFE returns at multiple rolling horizons and compare to mean implied vol per day. Gap = vol-risk-premium estimate.

In [ ]:
print("=== Q8: implied vs realized vol of VFE ===")
vfe = wide[["day", "timestamp", "S"]].sort_values(["day", "timestamp"]).copy()
vfe["ret"] = vfe.groupby("day")["S"].pct_change()
# realized vol over horizons (annualised, 100 ticks/sec assumption: timestamp step=100)
# 1 day = 1e6 ts/100 = 10000 ticks; we approximate as 10000 ticks/day, 7 days/yr (TTE basis)
TICKS_PER_DAY = 10000
TRADING_DAYS_YEAR = 252
def ann_vol(s, h):
    rolling = s.rolling(h).std() * np.sqrt(TICKS_PER_DAY * TRADING_DAYS_YEAR)
    return rolling.mean()
rv = {h: ann_vol(vfe["ret"], h) for h in (10, 50, 200, 500)}
print("Realized vol (annualised, mean over series):", {k: round(v, 4) for k, v in rv.items()})
mean_iv = iv_df.dropna(subset=["iv"]).groupby("day")["iv"].mean()
print("Mean implied vol per day:"); print(mean_iv.round(4))

## Q9 — Dead OTM strikes (VEV_6000 / VEV_6500)

Confirm the deep-OTM strikes are pinned at the platform 0.5 mid floor with ghost trades printing at price 0.0 — i.e. no real counterparty exists.

In [ ]:
print("=== Q9: VEV_6000 / VEV_6500 deep-OTM check ===")
for v in DEAD:
    sub = raw[raw["product"] == v]["mid_price"].dropna()
    print(f"{v}: rows={len(sub)} mean={sub.mean():.3f} min={sub.min()} "
          f"max={sub.max()} unique<=3={sub.value_counts().head(3).to_dict()}")
trades_files = sorted(glob.glob(os.path.join(ROOT, "data/round_3/trades_round_3_day_*.csv")))
tr = pd.concat([pd.read_csv(f, sep=";") for f in trades_files], ignore_index=True)
for v in DEAD:
    sub = tr[tr["symbol"] == v]
    print(f"{v} trades: n={len(sub)} price_range=[{sub.price.min() if len(sub) else 'n/a'}, "
          f"{sub.price.max() if len(sub) else 'n/a'}]")

## Q10 — Voucher vs VFE lead-lag (brief)

Quick sanity check using VEV_5200 (most ATM) vs VFE returns at the 100-ts grid. If lag-0 dominates, no exploitable intra-product lead-lag at this resolution.

In [ ]:
print("=== Q10: brief voucher-vs-VFE lead-lag (corr at lags) ===")
# Use most-ATM voucher as proxy
proxy = "VEV_5200"
ll = (raw[raw["product"].isin([proxy, UND])]
      .pivot_table(index=["day", "timestamp"], columns="product", values="mid_price")
      .dropna()
      .sort_index())
ll["dS"] = ll[UND].pct_change()
ll["dV"] = ll[proxy].pct_change()
for lag in (-3, -2, -1, 0, 1, 2, 3):
    c = ll["dV"].corr(ll["dS"].shift(lag))
    sign = "VFE leads V" if lag > 0 else ("V leads VFE" if lag < 0 else "concurrent")
    print(f"  lag={lag:+d} ({sign}): corr={c:.4f}")

## Summary + Key Findings

**Conventions**: TTE 8/7/6 days for historical day 0/1/2; year basis 365; smile `iv = a*m^2 + b*m + c` with `m = log(K/S)/sqrt(TTE_y)`.

### TL;DR
- **Smile drifts intraday — strongly.** Per-tick `(a, b, c)` refit shows curvature `a` swinging between roughly **-0.21 and +0.28** within day 1; `b` flips sign repeatedly (`mean ~ 0` so its std/|mean| explodes). Level `c` is the most stable (intraday `std/mean ~ 2-3%`, ranges ~0.21 to 0.25). CMU smoking-gun pattern.
- **Rolling smile (window=100) beats pooled static smile by 30.6% RMSE overall.** Per-strike: VEV_5000 +57%, VEV_5100 +47%, VEV_5200 +43%, VEV_5300 +35%, VEV_5500 +18%, VEV_5400 +0.6%. Static marginally better for VEV_4000 / VEV_4500 (deep ITM — RMSE governed by 1-tick rounding, not smile).
- **Residuals are stationary** (ADF stat -6 to -90, p ~ 0 for every live voucher). `mid - theo` mean-reverts -> exploitable alpha.
- **Persistent per-strike bias under rolling smile:**
  - VEV_5300 sits **+1.83 +/- 0.90** rich.
  - VEV_5400 sits **-1.98 +/- 0.73** cheap.
  - VEV_5200 sits **+1.06 +/- 0.73** rich, VEV_5000 **-0.46 +/- 0.56** cheap.
  - Offsets exceed typical 1-2 tick spread -> durable cross-strike RV trade (sell 5300/5200, buy 5400/5000, near delta-neutral butterfly).
- **Implied < Realized**: mean IV across days = **0.249 / 0.252 / 0.252**; realized vol of VFE at 10/50/200/500 ticks = **0.335 / 0.341 / 0.342 / 0.342** (annualised). Vouchers underprice realised vol by ~9 vol-points -> systematic long-vol / long-gamma EV-positive (subject to theta).

### Q1 — Per-voucher IV (count / mean / std)
| Voucher | n | mean IV | std |
|---|---|---|---|
| VEV_4000 | 2898 | 0.786 | 0.080 |
| VEV_4500 | 8823 | 0.450 | 0.043 |
| VEV_5000 | 29976 | 0.233 | 0.008 |
| VEV_5100 | 30000 | 0.231 | 0.008 |
| VEV_5200 | 30000 | 0.233 | 0.006 |
| VEV_5300 | 30000 | 0.236 | 0.006 |
| VEV_5400 | 30000 | 0.221 | 0.007 |
| VEV_5500 | 30000 | 0.240 | 0.006 |
| VEV_6000/6500 | - | - | pinned 0.5, no IV |

VEV_4000/4500 are deep ITM -> mid is essentially intrinsic, IV reflects rounded extrinsic ~1 tick -> noisy/biased. Treat as delta-1 proxies, not IV instruments.

### Q2 — Smile drift (per-tick refit)
- Day 0: `a` mean **0.077 +/- 0.056** (range **-0.08 -> +0.18**); `c` mean **0.229 +/- 0.005**.
- Day 1: `a` mean **0.057 +/- 0.076** (range **-0.21 -> +0.28**); `c` mean **0.230 +/- 0.006**.
- Day 2: `a` mean **0.066 +/- 0.070** (range **-0.19 -> +0.28**); `c` mean **0.228 +/- 0.007**.
- Curvature `a` swings sign intraday repeatedly. Level `c` stable to ~3%.
- Plot: `docs/round_3/research/plots/03_smile_drift.png`.

### Q3 — Static day-0 smile, out-of-sample
- Pooled day-0 fit: `a=0.1484, b=-0.0139, c=0.2260`.
- OOS residual means on day 1 / day 2 stay within +/-3.5 across strikes (e.g. VEV_5300: **+1.90 / +3.47 / +2.88**). Static generalises in level, but per-strike biases of 3+ ticks > typical spread -> Frankfurt's hardcoded smile would systematically misprice 5300 rich and 5400 cheap.

### Q4 — Rolling vs static
| Voucher | static RMSE | rolling RMSE | improvement |
|---|---|---|---|
| VEV_4000 | 1.27 | 1.63 | -28% |
| VEV_4500 | 0.75 | 0.98 | -31% |
| VEV_5000 | 1.67 | 0.72 | **+57%** |
| VEV_5100 | 1.68 | 0.89 | **+47%** |
| VEV_5200 | 2.27 | 1.29 | **+43%** |
| VEV_5300 | 3.12 | 2.04 | **+35%** |
| VEV_5400 | 2.12 | 2.11 | +0.6% |
| VEV_5500 | 0.70 | 0.57 | +18% |
| **Overall** | **2.01** | **1.40** | **+31%** |

Rolling wins decisively for ATM strikes (5000-5300). For deep ITM (4000/4500), RMSE is dominated by tick rounding so both fits are close; static marginally better.

### Q5 — Residual stationarity (rolling-smile residuals)
ADF stat in **[-93, -6]**, p ~ 0 for every live voucher -> strongly stationary. Per-voucher std of residual is **0.6-1.1**, so a **+/-1.5 threshold** gives a sensible entry trigger after vega-normalisation.

### Q6 — Cointegration
**Of 28 pairs, 8 are cointegrated at p<0.05.** Strongest non-trivial pairs:
- VEV_5400-VEV_5500 (**p=0.0008**)
- VEV_5300-VEV_5500 (**p=0.009**)
- VEV_5000-VEV_5500 (**p=0.018**)
- VEV_4000-VEV_4500 (**p ~ 0**) — intrinsic-driven, not real cointegration.
VEV_5500 is the common factor for the upper wing -> supports a butterfly / calendar of `5300 / 5400 / 5500` against the persistent bias from Q5.

### Q7 — Vega-weighted residuals
Per-voucher vega (mean): **5200=2.74, 5300=2.78, 5400=1.92, 5100=1.88, 5500=1.11, 5000=0.91, 4500=0.13, 4000=0.12.** Residual / vega:
- VEV_5400 = **-1.04** / vega-tick (cheapest)
- VEV_5300 = **+0.67** (richest)
Frankfurt-style threshold: `|resid| > k * vega` with **k ~ 0.5**.

### Q8 — Implied vs realized
| Horizon (ticks) | Realized vol (annualised) |
|---|---|
| 10 | 0.335 |
| 50 | 0.341 |
| 200 | 0.342 |
| 500 | 0.342 |

Mean IV by day: **0.249 / 0.252 / 0.252**. **Vouchers underprice realised vol by ~9 vol-points (~36% relative).** Naive long-straddle hedged would be EV-positive in expectation, but only if executable; bid-ask + theta will erase a chunk.

### Q9 — Dead OTM
- VEV_6000 mid: **30000 / 30000 rows = exactly 0.500**. Trades: **284, all at price 0.0**.
- VEV_6500 same: **30000 mids of 0.500, 284 trades all at 0.0**.
- **Verdict: completely dead. Exclude entirely from fitting and trading.** 0.5 mid is the platform floor with no real counterparty (ghost trades).

### Q10 — Lead-lag (brief, defer to cross-product agent)
VEV_5200 vs VFE returns at 100-ts grid: **lag-0 corr = 0.72**, all other lags within **+/-0.01**. No exploitable lead-lag at this resolution.

### Recommendations for our R3 voucher trader
1. **Use rolling smile, not Frankfurt-style hardcoded.** Window 100 ticks on per-tick `(a,b,c)` quadratic. Reduces RMSE by 31% overall, 40-57% on the strikes that matter.
2. **Universe**: trade **VEV_5000, 5100, 5200, 5300, 5400, 5500** only.
   - Skip VEV_6000 / VEV_6500 (dead-pinned at 0.5).
   - Treat VEV_4000 / VEV_4500 as delta-1 proxies for VFE; trade on mean-reversion of underlying or skip.
3. **Cross-strike RV bias to monetise** (from rolling-smile residuals):
   - VEV_5300 persistently rich by ~+1.8, VEV_5400 cheap by ~-2.0 -> fade 5300, lift 5400.
   - Delta-near-neutral; sizing capped by 300/voucher position limit.
4. **Vega gate** entry at `|resid| > 0.5 * vega` (Frankfurt pattern).
5. **Don't delta hedge** through VFE. Spread is 1-2 ticks on VFE; net delta from 5300/5400 butterfly small. Spend hedge budget on tighter quotes instead.
6. **IV-vs-RV gap is large but stale**: -9 vol points. Don't blindly trade vol; gap is partly theta compensation for 6-8d expiry. Ship residual-mean-reversion alpha first, layer vol exposure later.

### Plots
- `docs/round_3/research/plots/03_iv_timeseries.png` — per-voucher IV time series (3 days).
- `docs/round_3/research/plots/03_iv_surface.png` — mean IV (day x voucher) heatmap.
- `docs/round_3/research/plots/03_smile_drift.png` — `a(t), b(t), c(t)` per day.
- `docs/round_3/research/plots/03_residuals_per_strike.png` — rolling-smile residual series.